# Nominal vs. Real Revenue Exploration

Quick inspection of nominal vs. CPI-adjusted real revenue growth.

In [16]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px

# Set up BigQuery client
PROJECT_ID = "superstore-analysis-496710"
LOCATION = "EU"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# Query to fetch data from BigQuery
sql = """
SELECT
    order_month,
    order_count,
    customer_count,
    units_sold,
    units_per_order,
    nominal_revenue,
    real_revenue,
    nominal_revenue_index_jan_2021_100,
    real_revenue_index_jan_2021_100,
    nominal_avg_order_value,
    real_avg_order_value,
    inflation_adjustment_factor,
    cpi_index_jan_2021_100,
    cpi_mom_rate,
    cpi_yoy_rate
FROM `superstore-analysis-496710.dbt_doruk.fct_monthly_revenue`
ORDER BY order_month
"""

# Execute the query and convert the result to a DataFrame
monthly_revenue = client.query(sql).to_dataframe()
monthly_revenue.head()

/Users/doruk/dev/3a-superstore-analysis/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,order_month,order_count,customer_count,units_sold,units_per_order,nominal_revenue,real_revenue,nominal_revenue_index_jan_2021_100,real_revenue_index_jan_2021_100,nominal_avg_order_value,real_avg_order_value,inflation_adjustment_factor,cpi_index_jan_2021_100,cpi_mom_rate,cpi_yoy_rate
0,2021-01-01,332774,96537,7482261,22.484512,262374739.350000000,262374739.350000000,100.000000000,100.000000000,788.447232506,788.447232506,1.000000000,100.000000000,0.016818209,0.149736813
1,2021-02-01,300211,94975,6752817,22.493570,245348178.100000000,243140821.269431865,93.510594300,92.669295000,817.252459437,809.899774723,0.991003166,100.907851200,0.009078512,0.156109102
2,2021-03-01,332385,96373,7469408,22.472157,280547713.260000000,275065691.149026038,106.926342800,104.836956400,844.044446230,827.551457344,0.980459573,101.992986600,0.010753726,0.161902437
3,2021-04-01,321862,96019,7233028,22.472451,281199642.310000000,271152269.984802484,107.174815300,103.345417600,873.665242588,842.448844488,0.964269612,103.705435400,0.016789869,0.171401536
4,2021-05-01,332218,96317,7482198,22.521952,300710698.340000000,287412348.077561296,114.611147100,109.542690300,905.160762933,865.131775153,0.955776930,104.626923800,0.008885633,0.165928531


In [17]:
# Drop last two anomalous months (July & August 2023)
monthly_revenue = monthly_revenue[:-2]

### Visualizing nominal vs. real revenue growth over time

In [18]:
# Create a line chart for nominal and real revenue growth index over time
fig = px.line(monthly_revenue, x='order_month', y=['nominal_revenue_index_jan_2021_100', 'real_revenue_index_jan_2021_100', 'cpi_index_jan_2021_100'], 
              labels={'value': 'Index (Jan 2021 = 100)', 'order_month': 'Order Month'}, 
              title='Nominal vs Real Revenue Growth and CPI Over Time')
fig.add_hline(y=100, line_dash='dash', line_color='gray')
fig.show()

In [19]:
# Line chart of nominal and real revenue over time
fig = px.line(monthly_revenue, x='order_month', y=['nominal_revenue', 'real_revenue'], 
              labels={'value': 'Revenue', 'order_month': 'Order Month'}, 
              title='Nominal vs Real Revenue Over Time')
fig.show()

In [20]:
# Create a line chart for customer counts over time
fig = px.line(monthly_revenue, x="order_month", y="customer_count",
              labels={"value": "Customer Count", "order_month": "Order Month"},
              title="Customer Count Over Time")

fig.show()

In [21]:
# Create a line chart for order counts over time
fig = px.line(monthly_revenue, x="order_month", y="order_count",
              labels={"value": "Order Count", "order_month": "Order Month"},
              title="Order Count Over Time")

fig.show()

In [22]:
# Create a line chart for units sold over time
fig = px.line(monthly_revenue, x="order_month", y="units_sold",
              labels={"value": "Units Sold", "order_month": "Order Month"},
              title="Units Sold Over Time")

fig.show()

## Dashboard-ready mart exploration

Quick checks for the focused marts that will feed the BI dashboard.

In [23]:
trend_sql = """
SELECT
    order_month,
    month_label,
    nominal_revenue,
    real_revenue,
    nominal_revenue_index_jan_2021_100,
    real_revenue_index_jan_2021_100,
    cpi_index_jan_2021_100
FROM `superstore-analysis-496710.dbt_doruk.mart_revenue_trend_monthly`
ORDER BY order_month
"""

kpi_sql = """
SELECT
    metric_key,
    base_month,
    comparison_month,
    base_value,
    comparison_value,
    absolute_change,
    pct_change,
    index_jan_2021_100
FROM `superstore-analysis-496710.dbt_doruk.mart_revenue_story_kpis`
ORDER BY metric_key
"""

revenue_trend = client.query(trend_sql).to_dataframe()
story_kpis = client.query(kpi_sql).to_dataframe()

revenue_trend.shape, story_kpis.shape

/Users/doruk/dev/3a-superstore-analysis/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


((30, 7), (6, 8))

In [24]:
display(revenue_trend.head())
display(story_kpis)

,order_month,month_label,nominal_revenue,real_revenue,nominal_revenue_index_jan_2021_100,real_revenue_index_jan_2021_100,cpi_index_jan_2021_100
0,2021-01-01,2021-01,262374739.350000000,262374739.350000000,100.000000000,100.000000000,100.000000000
1,2021-02-01,2021-02,245348178.100000000,243140821.269431865,93.510594300,92.669295000,100.907851200
2,2021-03-01,2021-03,280547713.260000000,275065691.149026038,106.926342800,104.836956400,101.992986600
3,2021-04-01,2021-04,281199642.310000000,271152269.984802484,107.174815300,103.345417600,103.705435400
4,2021-05-01,2021-05,300710698.340000000,287412348.077561296,114.611147100,109.542690300,104.626923800


,metric_key,base_month,comparison_month,base_value,comparison_value,absolute_change,pct_change,index_jan_2021_100
0,cpi_index,2021-01-01,2023-06-01,100.000000000,263.313851500,163.313851500,1.633138515,263.313851500
1,customer_count,2021-01-01,2023-06-01,96537.000000000,96020.000000000,-517.000000000,-0.005355460,99.464454000
2,nominal_revenue,2021-01-01,2023-06-01,262374739.350000000,533449438.900000000,271074699.550000000,1.033158528,203.315852800
3,order_count,2021-01-01,2023-06-01,332774.000000000,320640.000000000,-12134.000000000,-0.036463185,96.353681500
4,real_revenue,2021-01-01,2023-06-01,262374739.350000000,202590724.383685655,-59784014.966314345,-0.227857358,77.214264200
5,units_sold,2021-01-01,2023-06-01,7482261.000000000,7211404.000000000,-270857.000000000,-0.036199887,96.380011300


In [25]:
trend_long = revenue_trend.melt(
    id_vars=["order_month", "month_label"],
    value_vars=[
        "nominal_revenue_index_jan_2021_100",
        "real_revenue_index_jan_2021_100",
        "cpi_index_jan_2021_100",
    ],
    var_name="metric_key",
    value_name="index_value",
)

trend_labels = {
    "nominal_revenue_index_jan_2021_100": "Nominal revenue",
    "real_revenue_index_jan_2021_100": "Real revenue",
    "cpi_index_jan_2021_100": "CPI",
}
trend_long["metric"] = trend_long["metric_key"].map(trend_labels)

fig = px.line(
    trend_long,
    x="order_month",
    y="index_value",
    color="metric",
    markers=True,
    labels={"order_month": "Month", "index_value": "Index (Jan 2021 = 100)", "metric": "Metric"},
    title="Revenue Growth vs CPI",
)
fig.add_hline(y=100, line_dash="dash", line_color="gray")
fig.show()

In [26]:
kpi_labels = {
    "order_count": "Orders",
    "units_sold": "Units sold",
    "customer_count": "Customers",
}
kpi_order = ["Orders", "Units sold", "Customers"]

kpi_plot = story_kpis.copy()
kpi_plot["metric"] = kpi_plot["metric_key"].map(kpi_labels)
kpi_plot["metric"] = pd.Categorical(kpi_plot["metric"], categories=kpi_order, ordered=True)
kpi_plot["pct_change_display"] = kpi_plot["pct_change"] * 100
kpi_plot = kpi_plot.sort_values("metric")

fig = px.bar(
    kpi_plot,
    x="metric",
    y="pct_change_display",
    text=kpi_plot["pct_change_display"].map(lambda value: f"{value:.1f}%"),
    labels={"metric": "Metric", "pct_change_display": "Change from Jan 2021 to Jun 2023 (%)"},
    title="Headline Change: January 2021 to June 2023",
)
fig.add_hline(y=0, line_color="gray")
fig.update_traces(textposition="outside")
fig.show()

## Product price inflation exploration

Quick checks for effective paid item prices and CPI-aligned product price marts.

In [27]:
product_price_trend_sql = """
SELECT
    order_month,
    month_label,
    line_count,
    item_count,
    units_sold,
    median_effective_paid_unit_price,
    median_real_effective_paid_unit_price,
    median_source_unit_price,
    median_effective_paid_price_index_jan_2021_100,
    median_real_effective_paid_price_index_jan_2021_100,
    median_source_unit_price_index_jan_2021_100,
    cpi_index_jan_2021_100
FROM `superstore-analysis-496710.dbt_doruk.mart_product_price_trend_monthly`
ORDER BY order_month
"""

product_price_kpis_sql = """
SELECT
    base_month,
    comparison_month,
    base_median_effective_paid_unit_price,
    comparison_median_effective_paid_unit_price,
    median_effective_paid_unit_price_absolute_change,
    median_effective_paid_unit_price_pct_change,
    median_effective_paid_unit_price_index_jan_2021_100,
    base_median_real_effective_paid_unit_price,
    comparison_median_real_effective_paid_unit_price,
    median_real_effective_paid_unit_price_pct_change,
    cpi_pct_change,
    eligible_item_count,
    items_with_increased_effective_paid_price,
    items_with_unchanged_effective_paid_price,
    items_with_decreased_effective_paid_price,
    items_with_increased_effective_paid_price_pct,
    avg_item_effective_paid_price_pct_change,
    median_item_effective_paid_price_pct_change
FROM `superstore-analysis-496710.dbt_doruk.mart_product_price_story_kpis`
"""

product_price_trend = client.query(product_price_trend_sql).to_dataframe()
product_price_kpis = client.query(product_price_kpis_sql).to_dataframe()

product_price_trend.shape, product_price_kpis.shape

/Users/doruk/dev/3a-superstore-analysis/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


((30, 12), (1, 18))

In [28]:
display(product_price_trend.head())
display(product_price_kpis)

,order_month,month_label,line_count,item_count,units_sold,median_effective_paid_unit_price,median_real_effective_paid_unit_price,median_source_unit_price,median_effective_paid_price_index_jan_2021_100,median_real_effective_paid_price_index_jan_2021_100,median_source_unit_price_index_jan_2021_100,cpi_index_jan_2021_100
0,2021-01-01,2021-01,1660379,26939,7465447,19.110000000,19.110000000,35.280000000,100.000000000,100.000000000,100.000000000,100.000000000
1,2021-02-01,2021-02,1496820,26939,6737903,19.770000000,19.592132592,35.100000000,103.453689200,102.522933500,99.489795900,100.907851200
2,2021-03-01,2021-03,1656075,26939,7452367,20.530000000,20.128835034,35.300000000,107.430664600,105.331423500,100.056689300,101.992986600
3,2021-04-01,2021-04,1604624,26939,7216595,21.230000000,20.471443863,35.150000000,111.093668200,107.124248400,99.631519300,103.705435400
4,2021-05-01,2021-05,1658944,26939,7464868,21.900000000,20.931514767,35.100000000,114.599686000,109.531736100,99.489795900,104.626923800


,base_month,comparison_month,base_median_effective_paid_unit_price,comparison_median_effective_paid_unit_price,median_effective_paid_unit_price_absolute_change,median_effective_paid_unit_price_pct_change,median_effective_paid_unit_price_index_jan_2021_100,base_median_real_effective_paid_unit_price,comparison_median_real_effective_paid_unit_price,median_real_effective_paid_unit_price_pct_change,cpi_pct_change,eligible_item_count,items_with_increased_effective_paid_price,items_with_unchanged_effective_paid_price,items_with_decreased_effective_paid_price,items_with_increased_effective_paid_price_pct,avg_item_effective_paid_price_pct_change,median_item_effective_paid_price_pct_change
0,2021-01-01,2023-06-01,19.110000000,40.190000000,21.080000000,1.103087389,210.308738900,19.110000000,15.263154517,-0.201300130,1.633138515,26939,26939,0,0,1.0,1.112580173,1.111202636


In [29]:
price_trend_long = product_price_trend.melt(
    id_vars=["order_month", "month_label"],
    value_vars=[
        "median_effective_paid_price_index_jan_2021_100",
        "median_real_effective_paid_price_index_jan_2021_100",
        "median_source_unit_price_index_jan_2021_100",
        "cpi_index_jan_2021_100",
    ],
    var_name="metric_key",
    value_name="index_value",
)

price_trend_labels = {
    "median_effective_paid_price_index_jan_2021_100": "Effective paid item price",
    "median_real_effective_paid_price_index_jan_2021_100": "Real effective paid item price",
    "median_source_unit_price_index_jan_2021_100": "Source unit price",
    "cpi_index_jan_2021_100": "CPI",
}
price_trend_long["metric"] = price_trend_long["metric_key"].map(price_trend_labels)

fig = px.line(
    price_trend_long,
    x="order_month",
    y="index_value",
    color="metric",
    markers=True,
    labels={"order_month": "Month", "index_value": "Index (Jan 2021 = 100)", "metric": "Metric"},
    title="Effective Paid Item Price vs CPI",
)
fig.add_hline(y=100, line_dash="dash", line_color="gray")
fig.show()

In [30]:
price_kpis = product_price_kpis.iloc[0]

summary_rows = [
    {
        "metric": "Median effective paid item price",
        "value": f"{price_kpis['median_effective_paid_unit_price_pct_change'] * 100:.1f}%",
    },
    {
        "metric": "Median real effective paid item price",
        "value": f"{price_kpis['median_real_effective_paid_unit_price_pct_change'] * 100:.1f}%",
    },
    {
        "metric": "CPI",
        "value": f"{price_kpis['cpi_pct_change'] * 100:.1f}%",
    },
    {
        "metric": "Items with increased effective paid price",
        "value": f"{price_kpis['items_with_increased_effective_paid_price_pct'] * 100:.1f}%",
    },
    {
        "metric": "Eligible item count",
        "value": f"{price_kpis['eligible_item_count']:,.0f}",
    },
]

pd.DataFrame(summary_rows)

,metric,value
0,Median effective paid item price,110.3%
1,Median real effective paid item price,-20.1%
2,CPI,163.3%
3,Items with increased effective paid price,100.0%
4,Eligible item count,"26,939"


## Category price inflation exploration

Category-level view of effective paid product price increases, using the dashboard mart at monthly `category1` grain.

In [31]:
category_price_trend_sql = """
SELECT
    order_month,
    month_label,
    category1,
    item_count,
    line_count,
    units_sold,
    median_effective_paid_unit_price,
    median_real_effective_paid_unit_price,
    cpi_index_jan_2021_100,
    median_effective_paid_price_index_jan_2021_100,
    median_real_effective_paid_price_index_jan_2021_100,
    median_effective_paid_price_pct_change_since_jan_2021,
    price_increase_rank
FROM `superstore-analysis-496710.dbt_doruk.mart_product_category_price_trend_monthly`
ORDER BY order_month, price_increase_rank
"""

category_price_trend = client.query(category_price_trend_sql).to_dataframe()
category_price_trend.shape

/Users/doruk/dev/3a-superstore-analysis/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(630, 13)

In [ ]:
comparison_month = pd.Timestamp("2023-06-01").date()

category_price_endpoint = (
    category_price_trend
    .loc[category_price_trend["order_month"] == comparison_month]
    .copy()
    .sort_values("price_increase_rank")
)
category_price_endpoint["price_increase_pct"] = (
    category_price_endpoint["median_effective_paid_price_pct_change_since_jan_2021"] * 100
)
category_price_endpoint["real_price_increase_pct"] = (
    (category_price_endpoint["median_real_effective_paid_price_index_jan_2021_100"] / 100 - 1) * 100
)
category_price_endpoint["cpi_increase_pct"] = (
    category_price_endpoint["cpi_index_jan_2021_100"] / 100 - 1
) * 100

category_price_endpoint[[
    "price_increase_rank",
    "category1",
    "price_increase_pct",
    "real_price_increase_pct",
    "cpi_increase_pct",
    "item_count",
    "line_count",
    "units_sold",   
]]

,price_increase_rank,category1,price_increase_pct,real_price_increase_pct,cpi_increase_pct,item_count,line_count,units_sold
609,1,SICAK ICECEKLER,121.119592900,-16.024321700,163.313851500,151,9009,40375
610,2,SARF,116.316440000,-17.848438700,163.313851500,143,8300,37453
611,3,TEMIZLIK,113.138324200,-19.055407400,163.313851500,1041,61684,279045
612,4,MANAV,112.911392400,-19.141590400,163.313851500,259,15481,69210
613,5,BEBEK,112.771233700,-19.194819200,163.313851500,869,51201,230337
614,6,SUT,112.629896100,-19.248495700,163.313851500,830,49167,221430
615,7,KAHVALTILIK,112.083656100,-19.455943900,163.313851500,1084,63962,287687
616,8,ET-TAVUK,111.997670400,-19.488599200,163.313851500,398,23592,105887
617,9,CAY-KAHVE-SEKER,111.920529800,-19.517895200,163.313851500,291,17380,78103
618,10,KOZMETIK,111.366245700,-19.728398400,163.313851500,3206,190069,854689


In [33]:
top_categories = category_price_endpoint.head(10)[[
    "price_increase_rank",
    "category1",
    "price_increase_pct",
    "real_price_increase_pct",
    "item_count",
]]

bottom_categories = category_price_endpoint.tail(10)[[
    "price_increase_rank",
    "category1",
    "price_increase_pct",
    "real_price_increase_pct",
    "item_count",
]]

display(top_categories.style.format({
    "price_increase_pct": "{:.1f}%",
    "real_price_increase_pct": "{:.1f}%",
    "item_count": "{:,.0f}",
}))
display(bottom_categories.style.format({
    "price_increase_pct": "{:.1f}%",
    "real_price_increase_pct": "{:.1f}%",
    "item_count": "{:,.0f}",
}))

,price_increase_rank,category1,price_increase_pct,real_price_increase_pct,item_count
609,1,SICAK ICECEKLER,121.1%,-16.0%,151
610,2,SARF,116.3%,-17.8%,143
611,3,TEMIZLIK,113.1%,-19.1%,"1,041"
612,4,MANAV,112.9%,-19.1%,259
613,5,BEBEK,112.8%,-19.2%,869
614,6,SUT,112.6%,-19.2%,830
615,7,KAHVALTILIK,112.1%,-19.5%,"1,084"
616,8,ET-TAVUK,112.0%,-19.5%,398
617,9,CAY-KAHVE-SEKER,111.9%,-19.5%,291
618,10,KOZMETIK,111.4%,-19.7%,"3,206"


,price_increase_rank,category1,price_increase_pct,real_price_increase_pct,item_count
620,12,OYUNCAK,111.1%,-19.8%,"2,232"
621,13,EV,110.8%,-19.9%,"9,831"
622,14,GIDA,110.6%,-20.0%,"3,108"
623,15,SOGUK ICECEKLER,110.5%,-20.1%,642
624,16,KAGIT,110.4%,-20.1%,299
625,17,SEKERLEME,110.3%,-20.1%,"1,538"
626,18,KARO,110.2%,-20.2%,38
627,19,DENIZ MAHSULLERI,109.5%,-20.4%,30
628,20,SIGARALAR,108.2%,-20.9%,93
629,21,MUHTELIF,106.9%,-21.4%,18


In [34]:
chart_df = category_price_endpoint.sort_values("price_increase_pct", ascending=True).copy()
cpi_reference = chart_df["cpi_increase_pct"].iloc[0]

fig = px.bar(
    chart_df,
    x="price_increase_pct",
    y="category1",
    orientation="h",
    text=chart_df["price_increase_pct"].map(lambda value: f"{value:.1f}%"),
    hover_data={
        "category1": False,
        "price_increase_pct": ":.1f",
        "real_price_increase_pct": ":.1f",
        "item_count": ":,",
        "line_count": ":,",
        "units_sold": ":,",
    },
    labels={
        "category1": "Category",
        "price_increase_pct": "Price increase from Jan 2021 to Jun 2023 (%)",
        "real_price_increase_pct": "Real price change (%)",
        "item_count": "Items",
        "line_count": "Lines",
        "units_sold": "Units sold",
    },
    title="Product Price Increase by Category: January 2021 to June 2023",
    height=max(520, 28 * len(chart_df)),
)
fig.add_vline(
    x=cpi_reference,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"CPI: {cpi_reference:.1f}%",
    annotation_position="top right",
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(xaxis_ticksuffix="%", yaxis_title=None, margin={"l": 140, "r": 60, "t": 80, "b": 60})
fig.show()